In [1]:
!pip install geemap earthengine-api -q

import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.0 MB/s eta 0:00:00


In [21]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='ee-pakjira')

In [80]:
# Load FAO GAUL (Administrative boundaries)
admin = ee.FeatureCollection("FAO/GAUL/2015/level1")

# Select Ang Thong Province
roi = admin.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', 'Thailand'),
        ee.Filter.eq('ADM1_NAME', 'Ang Thong')
    )
)

# Sentinel-2 Composite
image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi)
         .filterDate('2024-01-01', '2024-12-31')
         .median()
         .clip(roi))

# RGB Map
Map_rgb = geemap.Map()
Map_rgb.centerObject(roi, 9)

Map_rgb.addLayer(
    image,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000},
    "RGB"
)

Map_rgb.addLayer(roi, {}, "Ang Thong Boundary")

Map_rgb

Map(center=[14.62005629393108, 100.35163660829629], controls=(WidgetControl(options=['position', 'transparent_…

In [81]:
# Indices
ndvi = image.normalizedDifference(['B8','B4']).rename('NDVI')
ndwi = image.normalizedDifference(['B3','B8']).rename('NDWI')
ndbi = image.normalizedDifference(['B11','B8']).rename('NDBI')

# Feature stack
features = image.select(['B2','B3','B4','B8']).addBands([
    ndvi,
    ndwi,
    ndbi
])

# Map indices
Map_index = geemap.Map()
Map_index.centerObject(roi, 11)

Map_index.addLayer(ndvi, {'min':-1,'max':1,'palette':['blue','white','green']}, 'NDVI')
Map_index.addLayer(ndwi, {'min':-1,'max':1,'palette':['brown','white','blue']}, 'NDWI')
Map_index.addLayer(ndbi, {'min':-1,'max':1,'palette':['green','white','red']}, 'NDBI')

Map_index


Map(center=[14.62005629393108, 100.35163660829629], controls=(WidgetControl(options=['position', 'transparent_…

In [82]:
# Dynamic World labels
dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
      .filterBounds(roi)
      .filterDate('2023-01-01','2023-12-31')
      .select('label')
      .median()
      .clip(roi))

label = dw.toInt()

# จัด class
#1 Water, ,2 Wetland Vegetation, 3 Agriculture, 4 Forest, 5 Urban
remapped = label.remap(
    [0,3,4,1,6,2,5,7,8],
    [1,2,3,4,5,2,2,2,2]
).rename('class')

In [83]:
# Sampling
samples = features.addBands(remapped).sample(
    region=roi,
    scale=10,
    numPixels=6000,
    seed=10,
    geometries=True
)

samples = samples.filter(ee.Filter.notNull(['class']))

In [84]:
# Train/Test split
samples = samples.randomColumn('random')

train = samples.filter(ee.Filter.lt('random',0.8))
test = samples.filter(ee.Filter.gte('random',0.8))

In [85]:
# Random Forest (50 trees)
rf50 = ee.Classifier.smileRandomForest(
    numberOfTrees=50
).train(
    features=train,
    classProperty='class',
    inputProperties=features.bandNames()
)

rf_test = test.classify(rf50)

rf_matrix = rf_test.errorMatrix('class','classification')

print('RF Confusion Matrix:', rf_matrix.getInfo())
print('RF Overall Accuracy:', rf_matrix.accuracy().getInfo())
print('RF Kappa:', rf_matrix.kappa().getInfo())

producers = [i[0] for i in rf_matrix.producersAccuracy().getInfo()]
users = rf_matrix.consumersAccuracy().getInfo()[0]

print("Producer's Accuracy:", producers)
print("User's Accuracy:", users)

f1_scores = []

for p,u in zip(producers,users):
    if (p+u)==0:
        f1=0
    else:
        f1=2*(p*u)/(p+u)
    f1_scores.append(f1)

print("F1-score per class:",f1_scores)

RF Confusion Matrix: [[0, 0, 0, 0, 0, 0], [0, 70, 4, 46, 1, 3], [0, 8, 6, 82, 2, 17], [0, 6, 8, 618, 2, 62], [0, 0, 1, 15, 12, 12], [0, 3, 3, 84, 12, 156]]
RF Overall Accuracy: 0.6914556962025317
RF Kappa: 0.4429182153403165
Producer's Accuracy: [0, 0.5267175572519084, 0.05217391304347826, 0.888268156424581, 0.2972972972972973, 0.5735849056603773]
User's Accuracy: [0, 0.8117647058823529, 0.24, 0.7186440677966102, 0.4074074074074074, 0.628099173553719]
F1-score per class: [0, 0.6388888888888888, 0.08571428571428572, 0.7945034353529044, 0.34375, 0.5996055226824457]


In [86]:
# Random Forest Map
rf_classified = features.classify(rf50)

Map_rf = geemap.Map()
Map_rf.centerObject(roi,11)

Map_rf.addLayer(
    rf_classified,
    {'min':1,'max':5,'palette':['blue','cyan','yellow','green','red']},
    'Random Forest Classification'
)

# Legend
legend_keys = [
    'Water',
    'Wetland Vegetation',
    'Agriculture',
    'Forest',
    'Urban'
]
legend_colors = [
    (0, 0, 255),    # Cyan
    (0, 255, 255),  # Blue
    (255, 255, 0),  # Yellow
    (0, 128, 0),    # Green
    (255, 0, 0)     # Red
]

Map_rf.add_legend(title='Land Cover Classes', keys=legend_keys, colors=legend_colors)

Map_rf

Map(center=[14.62005629393108, 100.35163660829629], controls=(WidgetControl(options=['position', 'transparent_…

In [87]:
# Gradient Tree Boost
gtb = ee.Classifier.smileGradientTreeBoost(
    numberOfTrees=100,
    shrinkage=0.1,
    maxNodes=20
).train(
    features=train,
    classProperty='class',
    inputProperties=features.bandNames()
)

gtb_test = test.classify(gtb)

gtb_matrix = gtb_test.errorMatrix('class','classification')

print('GTB Confusion Matrix:',gtb_matrix.getInfo())
print('GTB Overall Accuracy:',gtb_matrix.accuracy().getInfo())
print('GTB Kappa:',gtb_matrix.kappa().getInfo())

producers = [i[0] for i in gtb_matrix.producersAccuracy().getInfo()]
users = gtb_matrix.consumersAccuracy().getInfo()[0]

print("Producer's Accuracy:", producers)
print("User's Accuracy:", users)

f1_scores=[]

for p,u in zip(producers,users):
    if (p+u)==0:
        f1=0
    else:
        f1=2*(p*u)/(p+u)
    f1_scores.append(f1)

print("F1-score per class:",f1_scores)


GTB Confusion Matrix: [[0, 0, 0, 0, 0, 0], [0, 70, 6, 51, 0, 4], [0, 6, 5, 88, 3, 13], [0, 6, 5, 644, 2, 59], [0, 0, 0, 17, 11, 9], [0, 6, 1, 92, 9, 157]]
GTB Overall Accuracy: 0.7017405063291139
GTB Kappa: 0.45878980472148334
Producer's Accuracy: [0, 0.5343511450381679, 0.043478260869565216, 0.8994413407821229, 0.2972972972972973, 0.5924528301886792]
User's Accuracy: [0, 0.7954545454545454, 0.29411764705882354, 0.7219730941704036, 0.44, 0.6487603305785123]
F1-score per class: [0, 0.6392694063926941, 0.07575757575757575, 0.8009950248756218, 0.3548387096774193, 0.6193293885601577]


In [89]:
# GTB Map
gtb_classified = features.classify(gtb)

Map_gtb = geemap.Map()
Map_gtb.centerObject(roi,11)

Map_gtb.addLayer(
    gtb_classified,
    {'min':1,'max':5,'palette':['blue','cyan','yellow','green','red']},
    'Gradient Tree Boost Classification'
)

Map_gtb.add_legend(title='Land Cover Classes', keys=legend_keys, colors=legend_colors)

Map_gtb

Map(center=[14.62005629393108, 100.35163660829629], controls=(WidgetControl(options=['position', 'transparent_…

In [96]:
# Export ข้อมูล
rf50_classified = features.classify(rf50)

task_rf = ee.batch.Export.image.toDrive(
    image=rf50_classified,
    description='RF50_Land_cover',
    folder='GEE_export',
    fileNamePrefix='RF50_land_cover',
    region=roi,
    scale=10,
    maxPixels=1e13
)

task_rf.start()
print("RF50 export started")

RF50 export started


In [97]:
gtb_classified = features.classify(gtb)

task_gtb = ee.batch.Export.image.toDrive(
    image=gtb_classified,
    description='GTB_Land_cover',
    folder='GEE_export',
    fileNamePrefix='GTB_land_cover',
    region=roi,
    scale=10,
    maxPixels=1e13
)

task_gtb.start()
print("GTB export started")

GTB export started
